# Assignment 1 — Q1: Online Bookstore Data Wrangling Pipeline (books.toscrape.com)
**Data Science Lab (CS2311)**

**Pipeline:** web pages (HTML) → Python dictionaries → **JSON** (semi-structured) → **Pandas DataFrame** (structured) → cleaning → feature engineering → normalization → visualization → CSV + business insights.

| Part | What we do |
|---|---|
| (a) | Scrape all 1,000 books (50 listing pages × 20 books, plus each book's own page) |
| (b) | Extract 14 attributes per book; save as JSON; load into a DataFrame |
| (c) | Missing values, duplicates, text cleanup, type conversion, 6 new features, Min-Max normalization |
| (d) | 7 visualizations |
| (e) | Save CSV + 5 business insights |

**Three data shapes:**
- **Unstructured (HTML):** text mixed with tags; a program must parse it before it can use it.
- **Semi-structured (JSON):** labelled key–value records; flexible, records may have different fields.
- **Structured (DataFrame / CSV):** fixed columns, one row per record, one data type per column.

In [ ]:
import json
import os
import re
import time
import warnings
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from bs4 import BeautifulSoup
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

---
## (a) Scraping

### How the site is organised (open it in a browser and use "Inspect" to see this)

- **Listing pages:** `https://books.toscrape.com/catalogue/page-1.html` … `page-50.html`, 20 books each.
- Each book on a listing page is an `<article class="product_pod">` containing:
  - `<p class="star-rating Three">` → the rating is the **second class name**
  - `<h3><a href="..." title="Full Title">Short title...</a></h3>` → the full title is in the `title` **attribute** (the visible text is cut short)
  - `<p class="price_color">£51.77</p>`
  - `<p class="instock availability"> In stock </p>`
- **Book page** (follow the link): category in the breadcrumb (`ul.breadcrumb`), description in the `<p>` right after `<div id="product_description">`, and a **product information table** with UPC, product type, prices, tax, availability and number of reviews.

**Analogy:** the listing page is a shop shelf showing the cover and price; the book page is the back cover with all the details. We read the shelf first, then pick up every book.

In [ ]:
BASE_URL = 'https://books.toscrape.com/'
HEADERS = {'User-Agent': 'Mozilla/5.0 (educational scraping exercise)'}
RAW_JSON = 'books_raw.json'
MAX_PAGES = 50          # 50 pages x 20 = 1000 books (all available). Lower it to test quickly.
session = requests.Session()
session.headers.update(HEADERS)


def get_soup(url):
    """Download a page and parse it. .content (bytes) lets BeautifulSoup detect the encoding, so '£' stays '£'."""
    response = session.get(url, timeout=30)
    response.raise_for_status()               # raise an error for 404 / 500 instead of silently continuing
    return BeautifulSoup(response.content, 'html.parser')


def safe_text(parent, selector):
    """Text of the first element matching a CSS selector, or None if it does not exist."""
    el = parent.select_one(selector)
    return el.get_text(strip=True) if el else None

### Step 1 — parse one listing page

In [ ]:
def parse_listing_page(page_url):
    soup = get_soup(page_url)
    books = []
    for article in soup.select('article.product_pod'):
        link = article.h3.a
        rating_tag = article.select_one('p.star-rating')
        image = article.select_one('div.image_container img')
        books.append({
            'title': link.get('title'),
            'price': safe_text(article, 'p.price_color'),
            'rating': rating_tag['class'][1] if rating_tag else None,        # ['star-rating', 'Three'] -> 'Three'
            'availability': safe_text(article, 'p.availability'),
            'product_url': urljoin(page_url, link['href']),                  # relative -> absolute URL
            'image_url': urljoin(page_url, image['src']) if image else None,
        })
    next_link = soup.select_one('li.next a')
    next_url = urljoin(page_url, next_link['href']) if next_link else None
    return books, next_url

first_page_books, next_url = parse_listing_page(urljoin(BASE_URL, 'catalogue/page-1.html'))
print('Books on page 1:', len(first_page_books), '| next page:', next_url)
first_page_books[0]

### Step 2 — parse one book's own page

The product information table is turned into a dictionary in one line: each row has a `<th>` (label) and a `<td>` (value).

In [ ]:
def parse_book_page(book_url):
    soup = get_soup(book_url)
    info = {row.th.get_text(strip=True): row.td.get_text(strip=True)
            for row in soup.select('table.table tr')}
    breadcrumb = soup.select('ul.breadcrumb li a')          # [Home, Books, <Category>]
    desc_header = soup.select_one('div#product_description')
    description = desc_header.find_next_sibling('p').get_text(strip=True) if desc_header else None
    return {
        'category': breadcrumb[2].get_text(strip=True) if len(breadcrumb) > 2 else None,
        'description': description,
        'upc': info.get('UPC'),
        'product_type': info.get('Product Type'),
        'price_excl_tax': info.get('Price (excl. tax)'),
        'price_incl_tax': info.get('Price (incl. tax)'),
        'tax': info.get('Tax'),
        'stock': info.get('Availability'),                   # e.g. 'In stock (22 available)'
        'num_reviews': info.get('Number of reviews'),
    }

parse_book_page(first_page_books[0]['product_url'])

### Step 3 — scrape everything

We follow the "next" button from page to page, then visit every book page. That is 50 + 1,000 = 1,050 requests, which takes a few minutes. A short pause between listing pages is polite to the server.

**Checkpoint:** the raw records are saved to `books_raw.json`. If the file already exists, we load it instead of scraping again, so re-running the notebook is fast. Delete the file to force a fresh scrape.

In [ ]:
def scrape_books(max_pages=MAX_PAGES):
    url = urljoin(BASE_URL, 'catalogue/page-1.html')
    all_books, page = [], 0
    while url and page < max_pages:
        books, url = parse_listing_page(url)
        page += 1
        for book in books:
            try:
                book.update(parse_book_page(book['product_url']))
            except requests.RequestException as e:
                print('  could not open', book['product_url'], '->', e)
        all_books.extend(books)
        if page % 5 == 0:
            print(f'page {page:2d}: {len(all_books):4d} books so far')
        time.sleep(0.2)
    return all_books

if os.path.exists(RAW_JSON):
    print(f'Loading cached {RAW_JSON} (delete it to scrape again)')
else:
    start = time.time()
    records = scrape_books()
    print(f'Scraped {len(records)} books in {time.time() - start:.0f} s')

    # ---------- (b) HTML -> JSON (semi-structured) ----------
    with open(RAW_JSON, 'w', encoding='utf-8') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

---
## (b) JSON → DataFrame

The JSON file is a **list of dictionaries**; `pd.DataFrame` turns each dictionary into a row and each key into a column. We extracted **15 attributes**: title, price, rating, availability, product URL, image URL, category, description, UPC, product type, price excl. tax, price incl. tax, tax, stock text and number of reviews.

In [ ]:
with open(RAW_JSON, encoding='utf-8') as f:
    raw = json.load(f)

print('Number of JSON records:', len(raw))
print('One record as stored in JSON:')
print(json.dumps(raw[0], indent=2, ensure_ascii=False)[:900])

df = pd.DataFrame(raw)
print('\nDataFrame shape:', df.shape)
df.head()

In [ ]:
df.info()

Every column is still **text** (`object`/`str`): '£51.77', 'Three', 'In stock (22 available)'. You cannot average text, so most of part (c) is about converting it.

---
## (c) Data wrangling
### c1. Missing values

**Strategy:**
- `description` is free text → fill with `'No description'` (a placeholder is honest; inventing text is not).
- Numeric columns → fill with the **median** after conversion (robust to outliers).
- Rows missing a key identifier (`title` or `upc`) → drop.

In [ ]:
print('Missing values (None) per column:\n', df.isnull().sum(), sep='')

In [ ]:
df = df.dropna(subset=['title', 'upc'])
df['description'] = df['description'].fillna('No description')
df['category'] = df['category'].fillna('Unknown')
print('Missing after text fills:', df[['title', 'upc', 'description', 'category']].isnull().sum().sum())

### c2. Duplicates
The **UPC** (Universal Product Code) uniquely identifies a book. Two rows with the same UPC are the same product scraped twice. Titles alone are not reliable: different books can share a title.

In [ ]:
print('Exact duplicate rows      :', df.duplicated().sum())
print('Duplicate UPCs            :', df['upc'].duplicated().sum())
print('Duplicate titles (allowed):', df['title'].duplicated().sum())
before = len(df)
df = df.drop_duplicates(subset='upc', keep='first').reset_index(drop=True)
print(f'Rows: {before} -> {len(df)}')

### c3. Inconsistent text formats
- Strip extra spaces, and collapse repeated whitespace/newlines inside descriptions.
- Category names: Title Case for consistency.
- Remove the "...more" suffix the site appends to long descriptions.

In [ ]:
text_cols = ['title', 'category', 'availability', 'product_type', 'description']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)

df['category'] = df['category'].str.title()
df['description'] = df['description'].str.replace(r'\s*\.\.\.more$', '', regex=True)
print('Number of categories:', df['category'].nunique())
df['category'].value_counts().head(10)

### c4. Converting to suitable data types

| Column | Raw text | Converted | How |
|---|---|---|---|
| price (and the 3 table prices) | '£51.77' | 51.77 (float) | remove everything except digits and the dot |
| rating | 'Three' | 3 (int) | dictionary mapping |
| stock | 'In stock (22 available)' | 22 (int) | regex `(\d+)` |
| num_reviews | '0' | 0 (int) | `pd.to_numeric` |

**Encoding note:** if a page is decoded wrongly, '£' can appear as 'Â£'. Removing every character that is not a digit or a dot handles both cases.

In [ ]:
def to_number(series):
    return pd.to_numeric(series.astype(str).str.replace(r'[^0-9.]', '', regex=True), errors='coerce')

for col in ['price', 'price_excl_tax', 'price_incl_tax', 'tax']:
    df[col] = to_number(df[col])

rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
df['rating'] = df['rating'].map(rating_map)

df['stock_count'] = pd.to_numeric(df['stock'].str.extract(r'(\d+)')[0], errors='coerce')
df['num_reviews'] = pd.to_numeric(df['num_reviews'], errors='coerce')

for col in ['price', 'rating', 'stock_count', 'num_reviews']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())
df['rating'] = df['rating'].astype(int)
df['stock_count'] = df['stock_count'].astype(int)
df['num_reviews'] = df['num_reviews'].astype(int)
df['category'] = df['category'].astype('category')

df[['price', 'rating', 'stock_count', 'num_reviews', 'category']].dtypes

### c5. Feature engineering (6 new features)

| Feature | Formula | Why it is useful |
|---|---|---|
| `price_category` | price bins: Budget (≤ £20), Mid-range (£20–40], Premium (> £40) | Easy segment for pricing strategy |
| `title_length` | number of characters in the title | Do long titles sell differently? |
| `description_words` | word count of the description | Richness of the product page |
| `availability_status` | 'High stock' / 'Medium stock' / 'Low stock' from stock_count | Inventory planning |
| `price_per_rating` | price / rating | "Cost per star": value-for-money indicator |
| `is_highly_rated` | 1 if rating ≥ 4 | Simple quality flag |

**Dry run of `pd.cut`** with bins [0, 20, 40, ∞]: price 51.77 falls in (40, ∞] → 'Premium'; price exactly 20 falls in (0, 20] → 'Budget' (the right edge is included).

In [ ]:
df['price_category'] = pd.cut(df['price'], bins=[0, 20, 40, np.inf], labels=['Budget', 'Mid-range', 'Premium'])
df['title_length'] = df['title'].str.len()
df['description_words'] = df['description'].str.split().str.len()
df['availability_status'] = pd.cut(df['stock_count'], bins=[-1, 5, 14, np.inf],
                                   labels=['Low stock', 'Medium stock', 'High stock'])
df['price_per_rating'] = (df['price'] / df['rating']).round(2)
df['is_highly_rated'] = (df['rating'] >= 4).astype(int)

df[['title', 'price', 'price_category', 'rating', 'price_per_rating', 'stock_count',
    'availability_status', 'title_length', 'description_words']].head()

### c6. Min-Max normalization

**Formula:** x' = (x − min) / (max − min), which maps every value into [0, 1].

Dry run: if min = 10, max = 60 and x = 35 → (35 − 10) / (60 − 10) = 0.5.

**Justification:** `price` (about £10–£60), `rating` (1–5) and `stock_count` (1–22) are on completely different scales. All three are **bounded** with **no extreme outliers** (see the box plots below), which is exactly when Min-Max works well: it puts them on one common 0–1 scale while keeping the shape of each distribution, so they can be compared or combined (e.g. in a score or a distance-based model). With strong outliers we would prefer StandardScaler or RobustScaler, because one extreme value would squash everything else near 0.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3))
for a, col in zip(ax, ['price', 'rating', 'stock_count']):
    sns.boxplot(x=df[col], ax=a, color='skyblue')
    a.set_title(f'{col}: range {df[col].min()} – {df[col].max()}')
plt.tight_layout()
plt.show()

scaler = MinMaxScaler()
norm_cols = ['price', 'rating', 'stock_count']
df[[c + '_norm' for c in norm_cols]] = scaler.fit_transform(df[norm_cols])
df[norm_cols + [c + '_norm' for c in norm_cols]].describe().loc[['min', 'max', 'mean']].round(3)

---
## (d) Visualizations

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sns.histplot(df['price'], bins=25, kde=True, ax=ax[0])
ax[0].axvline(df['price'].mean(), color='red', ls='--', label=f"mean £{df['price'].mean():.2f}")
ax[0].set(title='1. Price distribution', xlabel='Price (£)')
ax[0].legend()
sns.countplot(x='rating', data=df, ax=ax[1], palette='viridis')
ax[1].set(title='2. Rating distribution', xlabel='Star rating')
plt.tight_layout()
plt.show()

In [ ]:
top_categories = df['category'].value_counts().head(15)
plt.figure(figsize=(10, 6))
top_categories.sort_values().plot(kind='barh', color='tab:blue')
plt.title('3. Top 15 categories by number of books')
plt.xlabel('Number of books')
plt.show()

In [ ]:
cat_stats = (df.groupby('category', observed=True)
               .agg(books=('title', 'count'), avg_price=('price', 'mean'), avg_rating=('rating', 'mean'))
               .query('books >= 10')
               .sort_values('avg_price', ascending=False))
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
cat_stats['avg_price'].sort_values().plot(kind='barh', ax=ax[0], color='tab:orange')
ax[0].set(title='4a. Average price by category (≥ 10 books)', xlabel='Average price (£)')
cat_stats['avg_rating'].sort_values().plot(kind='barh', ax=ax[1], color='tab:green')
ax[1].set(title='4b. Average rating by category (≥ 10 books)', xlabel='Average rating')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(x='rating', y='price', data=df, palette='coolwarm')
plt.title('5. Price by star rating')
plt.xlabel('Star rating'); plt.ylabel('Price (£)')
plt.show()

In [ ]:
num_cols = ['price', 'rating', 'stock_count', 'title_length', 'description_words', 'price_per_rating', 'num_reviews']
corr = df[num_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('6. Correlation among numerical attributes')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
df['price_category'].value_counts().reindex(['Budget', 'Mid-range', 'Premium']).plot(
    kind='pie', autopct='%1.1f%%', ax=ax[0], colors=['#8dd3c7', '#ffffb3', '#fb8072'])
ax[0].set(title='7a. Share of books by price segment', ylabel='')
sns.histplot(df['stock_count'], bins=22, ax=ax[1], color='tab:purple')
ax[1].set(title='7b. Units in stock per title', xlabel='Units available')
plt.tight_layout()
plt.show()

---
## (e) Save the structured dataset and derive business insights

In [ ]:
output_cols = ['upc', 'title', 'category', 'price', 'price_excl_tax', 'price_incl_tax', 'tax', 'rating',
               'availability', 'stock_count', 'num_reviews', 'product_type', 'description', 'product_url',
               'image_url', 'price_category', 'title_length', 'description_words', 'availability_status',
               'price_per_rating', 'is_highly_rated', 'price_norm', 'rating_norm', 'stock_count_norm']
df[output_cols].to_csv('books_clean.csv', index=False, encoding='utf-8')
print('Saved books_clean.csv with shape', df[output_cols].shape)

### Numbers behind the insights
Every insight below is computed from the data, so it stays correct whatever the site shows on the day you scrape.

In [ ]:
r_price_rating = df['price'].corr(df['rating'])
mean_price_by_rating = df.groupby('rating')['price'].mean().round(2)
top_cat = df['category'].value_counts()
seg_share = df['price_category'].value_counts(normalize=True).mul(100).round(1)
high_rated_share = df['is_highly_rated'].mean() * 100
pricey = cat_stats['avg_price'].idxmax(), cat_stats['avg_price'].max()
cheap = cat_stats['avg_price'].idxmin(), cat_stats['avg_price'].min()
low_stock = (df['availability_status'] == 'Low stock').mean() * 100
best_value = df.sort_values(['rating', 'price'], ascending=[False, True]).head(5)[['title', 'category', 'price', 'rating']]

print(f'1. Correlation price vs rating: r = {r_price_rating:.3f}')
print('   Mean price by rating:', mean_price_by_rating.to_dict())
print(f'2. Largest categories: {top_cat.index[0]} ({top_cat.iloc[0]}), {top_cat.index[1]} ({top_cat.iloc[1]}), '
      f'{top_cat.index[2]} ({top_cat.iloc[2]}) out of {df.category.nunique()} categories')
print(f'3. Price segments (%): {seg_share.to_dict()}')
print(f'4. Most expensive category (≥10 books): {pricey[0]} £{pricey[1]:.2f}; cheapest: {cheap[0]} £{cheap[1]:.2f}')
print(f'5. Highly rated (4–5 stars): {high_rated_share:.1f}% | low-stock titles (≤5 units): {low_stock:.1f}%')
print(f'   Total reviews across all books: {df.num_reviews.sum()}')
print('\nBest-value picks (5 stars, lowest price):')
print(best_value.to_string(index=False))

### Analytical summary — five business insights

Read each one together with the printed numbers above.

1. **Price does not follow quality.** The correlation between price and rating is close to zero, and the average price is about the same for 1-star and 5-star books. Pricing is not quality-driven: the store could test a small premium on 4–5 star titles, and discount poorly rated expensive ones.
2. **The catalogue is concentrated in a few categories.** A handful of categories (shown in plot 3) hold a large share of all titles, while many categories have only one or two books. Marketing and stock planning should focus on the big categories; the long tail is a candidate for expansion or pruning.
3. **Prices are spread fairly evenly between about £10 and £60** (plot 1), and the three price segments are of similar size (insight 3 printout). There is no clear "price point" strategy: introducing deliberate price tiers per category could improve margins.
4. **Category price gaps exist** between the most and least expensive categories (insight 4 printout), which points to where premium positioning is already accepted by customers.
5. **Inventory and engagement data are weak.** Every title is listed as in stock, but a share of titles is at low stock (≤ 5 units) and need reordering. If the total number of reviews is 0 (as printed above), the store has no customer-feedback signal: encouraging reviews would give much better data for future pricing decisions.